In [23]:
from google.colab import userdata
import os
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [3]:
!kaggle datasets download -d muthuj7/weather-dataset

Dataset URL: https://www.kaggle.com/datasets/muthuj7/weather-dataset
License(s): CC0-1.0
  0% 0.00/2.23M [00:00<?, ?B/s]
100% 2.23M/2.23M [00:00<00:00, 164MB/s]


In [4]:
! unzip "weather-dataset.zip"

Archive:  weather-dataset.zip
  inflating: weatherHistory.csv      


In [5]:
import pandas as pd

In [5]:
data=pd.read_csv("weatherHistory.csv", parse_dates=['Formatted Date'])

In [ ]:
data.head()

,Formatted Date,Summary,Precip Type,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Loud Cover,Pressure (millibars),Daily Summary
0,2006-04-01 00:00:00+02:00,Partly Cloudy,rain,9.472222,7.388889,0.89,14.1197,251.0,15.8263,0.0,1015.13,Partly cloudy throughout the day.
1,2006-04-01 01:00:00+02:00,Partly Cloudy,rain,9.355556,7.227778,0.86,14.2646,259.0,15.8263,0.0,1015.63,Partly cloudy throughout the day.
2,2006-04-01 02:00:00+02:00,Mostly Cloudy,rain,9.377778,9.377778,0.89,3.9284,204.0,14.9569,0.0,1015.94,Partly cloudy throughout the day.
3,2006-04-01 03:00:00+02:00,Partly Cloudy,rain,8.288889,5.944444,0.83,14.1036,269.0,15.8263,0.0,1016.41,Partly cloudy throughout the day.
4,2006-04-01 04:00:00+02:00,Mostly Cloudy,rain,8.755556,6.977778,0.83,11.0446,259.0,15.8263,0.0,1016.51,Partly cloudy throughout the day.


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96453 entries, 0 to 96452
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype              
---  ------                    --------------  -----              
 0   Formatted Date            96453 non-null  object             
 1   Summary                   96453 non-null  object             
 2   Precip Type               95936 non-null  object             
 3   Temperature (C)           96453 non-null  float64            
 4   Apparent Temperature (C)  96453 non-null  float64            
 5   Humidity                  96453 non-null  float64            
 6   Wind Speed (km/h)         96453 non-null  float64            
 7   Wind Bearing (degrees)    96453 non-null  float64            
 8   Visibility (km)           96453 non-null  float64            
 9   Loud Cover                96453 non-null  float64            
 10  Pressure (millibars)      96453 non-null  float64            
 11  Daily Summary  

In [9]:
data['Date']=pd.to_datetime(data['Formatted Date'],utc=True)

In [ ]:
data.Date.max()
# pay attention to the date format

Timestamp('2016-12-31 22:00:00+0000', tz='UTC')

- Extract the average temperature last week

In [ ]:
data[data['Date']>='2016-12-24'].groupby('Date')['Temperature (C)'].mean().mean()

0.26439790575916244

- Find the days on which the humidity exceed 75%

In [ ]:
data[data['Humidity']>0.75]['Date'].dt.date.drop_duplicates()

,Date
0,2006-03-31
2,2006-04-01
26,2006-04-10
50,2006-04-11
74,2006-04-12
...,...
96311,2016-09-04
96335,2016-09-05
96359,2016-09-06
96383,2016-09-07


- Compare the average wind speed on sunny days versus rainy days

In [ ]:
data.groupby('Summary')['Wind Speed (km/h)'].mean()

,Wind Speed (km/h)
Summary,
Breezy,32.143948
Breezy and Dry,33.810000
Breezy and Foggy,33.477880
Breezy and Mostly Cloudy,33.386345
Breezy and Overcast,33.037566
Breezy and Partly Cloudy,33.532796
Clear,8.141352
Dangerously Windy and Partly Cloudy,63.852600
Drizzle,10.356428


In [ ]:
data.groupby('Precip Type')['Wind Speed (km/h)'].mean()


,Wind Speed (km/h)
Precip Type,
rain,10.971219
snow,9.481998


# Gemini

In [10]:
# prompt: Find the average temperature for the last seven days from the most recent date in the dataset.

import pandas as pd
data[data['Date']>=data.Date.max()-pd.DateOffset(days=7)]['Temperature (C)'].mean()


0.22692307692307706

In [12]:
# prompt: prompt: Display and compare the average wind speed on days

data.groupby('Summary')['Wind Speed (km/h)'].mean()


,Wind Speed (km/h)
Summary,
Breezy,32.143948
Breezy and Dry,33.810000
Breezy and Foggy,33.477880
Breezy and Mostly Cloudy,33.386345
Breezy and Overcast,33.037566
Breezy and Partly Cloudy,33.532796
Clear,8.141352
Dangerously Windy and Partly Cloudy,63.852600
Drizzle,10.356428


In [13]:
# prompt: - List all the days on which humidity is greater than 75%, return their corresponding humidity levels.

data[data['Humidity'] > 0.75][['Date','Humidity']]


,Date,Humidity
0,2006-03-31 22:00:00+00:00,0.89
1,2006-03-31 23:00:00+00:00,0.86
2,2006-04-01 00:00:00+00:00,0.89
3,2006-04-01 01:00:00+00:00,0.83
4,2006-04-01 02:00:00+00:00,0.83
...,...,...
96432,2016-09-09 01:00:00+00:00,0.87
96433,2016-09-09 02:00:00+00:00,0.93
96434,2016-09-09 03:00:00+00:00,0.90
96435,2016-09-09 04:00:00+00:00,0.93


In [17]:
# prompt: Analyze data, how can i understand if day rainy or sunny

# Assuming 'Summary' column contains information about rain or sun

# Count occurrences of rainy and sunny days
rainy_days = data[data['Summary'].str.contains('Rain', case=False)].shape[0]

# Use '|' to specify multiple patterns for sunny days
sunny_days = data[data['Summary'].str.contains('Sun|Clear', case=False)].shape[0]

print(f"Rainy days: {rainy_days}")
print(f"Sunny days: {sunny_days}")

# Calculate percentage of rainy and sunny days
total_days = data.shape[0]
rainy_percentage = (rainy_days / total_days) * 100
sunny_percentage = (sunny_days / total_days) * 100

print(f"Rainy days percentage: {rainy_percentage:.2f}%")
print(f"Sunny days percentage: {sunny_percentage:.2f}%")

# Explore other columns related to rain or sun, like 'Precip Type'
rainy_precip_types = data[data['Summary'].str.contains('Rain', case=False)]['Precip Type'].unique()
print(f"Precipitation types on rainy days: {rainy_precip_types}")

Rainy days: 73
Sunny days: 10890
Rainy days percentage: 0.08%
Sunny days percentage: 11.29%
Precipitation types on rainy days: ['rain']


In [1]:
!kaggle datasets download -d brendan45774/test-file

Dataset URL: https://www.kaggle.com/datasets/brendan45774/test-file
License(s): CC0-1.0
  0% 0.00/11.2k [00:00<?, ?B/s]
100% 11.2k/11.2k [00:00<00:00, 15.7MB/s]


In [2]:
! unzip "test-file.zip"

Archive:  test-file.zip
  inflating: tested.csv              


In [6]:
t_df=pd.read_csv("tested.csv")

In [7]:
t_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [8]:
# prompt: Using dataframe t_df: check how often we see james in name column

# Convert the 'Name' column to lowercase for case-insensitive matching.
t_df['Name'] = t_df['Name'].str.lower()

# Count occurrences of 'james' in the 'Name' column.
james_count = t_df['Name'].str.contains('james').sum()

# Print the result.
print(f'James appears in the Name column {james_count} times.')



James appears in the Name column 14 times.


To anonymize data using hashlib, follow these steps:

1. Import the hashlib Module: First, import hashlib into your Python script.

In [9]:
import hashlib

2. Choose a Hashing Algorithm: hashlib supports various hashing algorithms like sha256, md5, etc. sha256 is widely used for its balance between speed and security.

In [10]:
hashlib.sha256()

<sha256 _hashlib.HASH object @ 0x7be94d59fc50>

3. Hash the Data: Convert your data (e.g., a string or column from a dataset) into a hashed format. You need to encode the data to bytes before hashing.

In [11]:
def hash_data(data):
    return hashlib.sha256(data.encode()).hexdigest()

4. Apply to Dataset: Use the function to hash each item in the dataset’s column.

In [12]:
anonymized_dataset = [hash_data(name) for name in t_df]

In [15]:
anonymized_dataset[:5]

['c732e9cb18f0f9d000caca402481388ad6957cd4102dd8e3a5ec5910b5b2412d',
 '2028e21cf8189a8c8cf7b1e9bbf332a16c332d501732e082908bbe8fdadf62c2',
 'f3200e7fa9a3feca0de17e4c01e0d078e35ff8055824a705ffc1b76447580061',
 'dcd1d5223f73b3a965c07e3ff5dbee3eedcfedb806686a05b9b3868a2c3d6d50',
 '953dd6f2b46151da487be87d8d94a3aa11ffb822e179f07d697ebac46734cda6']

# Batch Processing and Asynchronous Calls

In [17]:
import pandas as pd

# Define batch size (number of rows to process at a time)
batch_size = 1000

# Create an empty DataFrame to accumulate results (if needed)
result_df = pd.DataFrame()

# Iterate through the dataset in batches
for batch_df in pd.read_csv("tested.csv", chunksize=batch_size):
    # Process each batch here
    # For demonstration, let's calculate the average age of passengers in each batch
    batch_average_age = batch_df['Age'].mean()

    # You can perform various analyses on each batch, e.g., calculate statistics

    # Accumulate or append results (if needed)
    # In this example, we'll append the batch average age to the result DataFrame
    result_df = result_df.append({"Batch Average Age": batch_average_age}, ignore_index=True)

# After processing all batches, you can work with the accumulated results (result_df)
# For example, you can calculate overall statistics or create visualizations.

AttributeError: 'DataFrame' object has no attribute 'append'

In [24]:
"""
Install the Google AI Python SDK

$ pip install google-generativeai
"""

import os
import google.generativeai as genai

genai.configure(api_key=os.environ["GEMINI_API_KEY"])

# Create the model
generation_config = {
  "temperature": 1,
  "top_p": 0.95,
  "top_k": 64,
  "max_output_tokens": 8192,
  "response_mime_type": "text/plain",
}

model = genai.GenerativeModel(
  model_name="gemini-1.5-pro",
  generation_config=generation_config,
  # safety_settings = Adjust safety settings
  # See https://ai.google.dev/gemini-api/docs/safety-settings
)

chat_session = model.start_chat(
  history=[
  ]
)

response = chat_session.send_message("INSERT_INPUT_HERE")

print(response.text)

Please provide me with the input you would like me to process. I need more context to understand what you want me to do. For example:

* **If you want me to complete a sentence or paragraph:** Provide the beginning of the text.
* **If you want me to answer a question:** Ask the question.
* **If you want me to generate creative content:** Describe the topic, style, and length. 

Once you provide more information, I'll be happy to help! 



In [25]:
!kaggle datasets download -d yasserh/titanic-dataset

Dataset URL: https://www.kaggle.com/datasets/yasserh/titanic-dataset
License(s): CC0-1.0
  0% 0.00/22.0k [00:00<?, ?B/s]
100% 22.0k/22.0k [00:00<00:00, 29.9MB/s]


In [26]:
! unzip "titanic-dataset.zip"

Archive:  titanic-dataset.zip
  inflating: Titanic-Dataset.csv     


In [32]:
df_titanic=pd.read_csv("Titanic-Dataset.csv")

In [29]:
chat_session = model.start_chat(
  history=[
  ]
)

In [31]:
response = chat_session.send_message("Analyze the correlation between passenger age and survival rates in the Titanic dataset, breaking down the analysis by age groups (e.g., children, adults, seniors). Provide statistical insights.")

print(response.text)

I can help you with that! Here's a breakdown of how we can analyze the correlation between passenger age and survival rates from the Titanic dataset, along with the statistical insights we can gain:

**1. Data Acquisition & Preparation**

* **Get the Data:** You'll need the Titanic passenger data, which is commonly available online (e.g., on Kaggle). This dataset usually contains information like passenger ID, age, sex, passenger class, fare, survival status, etc.
* **Data Cleaning:**
    * Handle missing values in the 'Age' column.  You can:
        * **Remove rows with missing ages:** This can bias the results if the missingness is not random.
        * **Impute ages:** Fill in missing values using the median or mean age (be cautious as this can also introduce bias). Consider imputing based on similar passenger profiles (e.g., class, sex).
    * Create an 'Age Group' categorical variable: Define age ranges that make sense for your analysis (e.g., Children (0-12), Teenagers (13-19), A